# Tasks:
1. Incoporate more slang, through datasets or looking through reviews
2. Include more stop words, getting token frequency and do a chart to justify
3. When including NER, I have to do it before basic cleaning, this is because basic cleaning removes punctuation making it difficult to determine sentances

# Load CSV

In [25]:
import pandas as pd

df = pd.read_csv("mcu_imdb_reviews.csv")

# Combine summary + content and remove NAs
df["summary"] = df["summary"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)
df["text"] = df["summary"] + " " + df["content"]

# Basic Cleaning


In [23]:
import re
import ftfy
from unidecode import unidecode

def basic_clean(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs, tho unlikey to have any
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # remove punctuation
    text = ftfy.fix_text(text) # fix encoding issues
    text = unidecode(text) # convert to ASCII - removes encoding artifacts
    
    return text

# Microtext Normalization

In [6]:
slang_dict = {
    "idk": "i do not know",
    "imo": "in my opinion",
    "luv": "love",
    "btw": "by the way",
    "10/10": "very good",
    "mcu": "marvel cinematic universe"
}

def normalize_slang(text):
    words = text.split()
    normalized = [slang_dict.get(word, word) for word in words]
    return " ".join(normalized)

# Tokenization and Lemmatization

In [7]:
import spacy
# Load English tokenizer, tagger, parser and NER
nlp = spacy.load("en_core_web_sm")

def tokenize_lemmatize(text):
    doc = nlp(text)
    tokens = []
    
    for token in doc:
        if not token.is_stop and not token.is_punct:
            tokens.append(token.lemma_)
    
    return tokens

# Pipeline

In [16]:
def preprocess(text):
    text = basic_clean(text)
    text = normalize_slang(text)
    tokens = tokenize_lemmatize(text)
    return " ".join(tokens)

# Applying onto dataset

In [9]:
df["processed_text"] = df["text"].apply(preprocess)

In [10]:
df["processed_text"]

0        good mcumovie attempt rewatch mcumovie chronol...
1        marvel superhero film class come rank marvel s...
2        not waste life stark blist superhero risky lea...
3        begin queue acdc marvel cinematic universe 202...
4        rdj iron man tony stark robert downey jr hard ...
                               ...                        
90305    feel like passive aggressiv pro life love cons...
90306    fantastic step mark significant reintroduction...
90307    direct matt shakman film focus spectacle emoti...
90308    fantastic step mark important new beginning ma...
90309    good version ff hesitant base previous one ve ...
Name: processed_text, Length: 90310, dtype: str

In [11]:
df.to_csv("Temp_preprocessed_data.csv", index=False)